# Rotation stop signal — global ratio vs marginal per-heading gain (TODO §4)

`RotationState.update(ratio)` stops rotating at a waypoint after `stop_after` consecutive
non-increasing values. Today it is fed the **global** coverage ratio
(`update_covered_mask` → `covered∩free / total_free`). Near completion one heading covers a few
local cells against a ~100k-cell denominator, so the marginal delta underflows into rounding
noise → a heading that *did* add cells reads as zero-gain → rotation stops early, leaving cells
uncovered.

**Hypothesis:** feeding `update` the raw **marginal cell count** (new cells this heading) instead
of the global ratio is the *same saturation logic with a more accurate input* — an optimisation,
not a behaviour change (same rule as the arrival-time-headings study). We A/B it before applying.

The two arms differ in **exactly one line**: what value goes into `RotationState.update`.
Everything else (drain-the-plan execution, arrival-time headings, `nearest_start`, `stop_after=3`)
is copied verbatim from `heading_timing_impact.ipynb` so the comparison is clean.

**Decision rule:** adopt the marginal-gain arm if it covers **≥** as much (no coverage regression)
with **≤** total headings on every map — i.e. it recovers the prematurely-skipped cells without
adding spins where the area really was saturated.

In [1]:
import sys, math
from collections import deque
from pathlib import Path
import numpy as np, yaml
from PIL import Image

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and not (ROOT / "docker-compose.yml").is_file():
    ROOT = ROOT.parent
ASSETS = ROOT / "assets"
sys.path.insert(0, str(ROOT / "ros2_ws/src/navigation/exploration"))

from exploration.explore_costmap_map import (
    build_map_data, compute_visibility, compute_headings_for_waypoint, update_covered_mask,
    pixel_to_world,
)
from exploration.execution_strategy import ExplorationSession
from exploration.rotation_strategy import get_headings, RotationState
print("repo root:", ROOT)

repo root: /home/idlab332/workspace/ros_ws/src/TRAVIS


In [2]:
def find_path(navigable_mask, start, goal):
    """BFS shortest 8-connected path (col,row) list; [start] if no path. Copy of demo_robot.find_path."""
    if start == goal:
        return [start]
    H, W = navigable_mask.shape
    q = deque([start]); came = {start: None}
    nb = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    while q:
        col, row = q.popleft()
        if (col, row) == goal:
            break
        for dc, dr in nb:
            nc, nr = col+dc, row+dr
            if 0 <= nc < W and 0 <= nr < H and navigable_mask[nr, nc] and (nc, nr) not in came:
                came[(nc, nr)] = (col, row); q.append((nc, nr))
    if goal not in came:
        return [start]
    path = [goal]
    while path[-1] != start:
        path.append(came[path[-1]])
    return path[::-1]


def load_map(name):
    d = ASSETS / name
    yml = yaml.safe_load(open(d / "map.yaml"))
    arr = np.array(Image.open(d / "map.pgm"), dtype=np.uint8)
    res = float(yml["resolution"])
    return build_map_data(1.0 - arr / 255.0, res, float(yml["origin"][0]),
                          float(yml["origin"][1]), inflation_radius_m=0.9)


def base_cfg():
    return {"max_detection_range": 6.0, "fov_horizontal": 87.0,
            "observation_rotation_increment": 30.0, "sampling_step_m": 3.0,
            "num_rays": 360, "frontier_weight": 1.0, "coverage_weight": 1.0,
            "travel_cost_weight": 1.0, "exploration_completion_threshold": 0.90,
            "planner_coverage_warning_threshold": 0.0}

## The simulated explore loop, parametrised by stop signal

Identical to `heading_timing_impact.simulate` (drain-the-plan, arrival-time headings) except the
rotation inner loop now computes **both** the global ratio and the marginal cell count each heading,
and feeds `RotationState.update` whichever the `stop_signal` arm selects:

- `stop_signal='ratio'`  → `update(global_ratio)`   — **current production behaviour**
- `stop_signal='gain'`   → `update(new_cells)`       — raw marginal count (proposed)

`RotationState`'s `value <= prev → increment` logic is unchanged; only the fed value differs.
Note `new_cells` is monotone ≥ 0, so `<= prev` still means “this heading added no more than the
previous one” — for a raw count that reduces to “added 0” only once gain hits 0 and stays there,
which is the intended saturation semantics. We keep `stop_after=3` for both arms.

In [3]:
def _arrival_headings(md, wp, cfg, max_range_px, current_heading):
    """Headings from the CURRENT covered_mask (same logic + sort as the node's get_headings path)."""
    cov_cells, _ = compute_visibility((wp.col, wp.row), md, max_range_px, cfg["num_rays"])
    headings = compute_headings_for_waypoint(
        wp.col, wp.row, cov_cells,
        fov_deg=cfg["fov_horizontal"], increment_deg=cfg["observation_rotation_increment"])
    if not headings:
        headings = [float(h) for h in range(0, 360, int(cfg["observation_rotation_increment"]))]
    return sorted((float(h) for h in headings), key=lambda h: (h - current_heading) % 360.0)


def simulate(md, cfg, stop_signal, start_cx, start_cy, max_iters=200):
    """Drain-the-plan explore loop (mirrors ros2_exploration_node). Arrival-time headings for both
    arms so the ONLY variable is the rotation stop signal.

    stop_signal in {'ratio','gain'} — value fed to RotationState.update.
    """
    md.covered_mask[:] = False
    H = md.pgm_array.shape[0]; res = md.resolution
    max_range_px = max(1, int(cfg["max_detection_range"] / res))
    fov = cfg["fov_horizontal"]; nrays = cfg["num_rays"]

    session = ExplorationSession(md, cfg)
    robot_col, robot_row = session.nearest_start(start_cx, start_cy)
    robot_x, robot_y = pixel_to_world(robot_col, robot_row, res, md.origin_x, md.origin_y, H)
    heading = 0.0
    total_spins = redundant_spins = waypoints_visited = 0
    final_ratio = 0.0
    # diagnostic: cells that a heading WOULD have added but were skipped because the arm stopped.
    cells_lost_to_early_stop = 0

    for _ in range(max_iters):
        waypoints, ratio, no_frontiers, _ = session.plan_waypoints_raw(robot_x, robot_y)
        final_ratio = ratio
        if (no_frontiers and ratio >= cfg["exploration_completion_threshold"]) or not waypoints:
            break

        for wp in waypoints:                       # DRAIN the plan
            path = find_path(md.navigable_mask, (robot_col, robot_row), (wp.col, wp.row))
            if session.on_unreachable(wp, path):
                continue
            robot_col, robot_row = wp.col, wp.row
            robot_x, robot_y = wp.x, wp.y

            rot_headings = _arrival_headings(md, wp, cfg, max_range_px, heading)
            rot = RotationState(stop_after=3)
            stopped = False
            for h in rot_headings:
                if stopped:
                    # arm already said stop; measure what these remaining headings WOULD have added
                    before = int(np.sum(md.covered_mask & md.free_mask))
                    update_covered_mask(md, wp.col, wp.row, float(h), fov, max_range_px, nrays)
                    cells_lost_to_early_stop += int(np.sum(md.covered_mask & md.free_mask)) - before
                    continue
                heading = float(h)
                before = int(np.sum(md.covered_mask & md.free_mask))
                ratio = update_covered_mask(md, wp.col, wp.row, heading, fov, max_range_px, nrays)
                after = int(np.sum(md.covered_mask & md.free_mask))
                new_cells = after - before
                total_spins += 1
                if new_cells == 0:
                    redundant_spins += 1
                signal = ratio if stop_signal == "ratio" else float(new_cells)
                if not rot.update(signal):
                    stopped = True   # keep looping only to measure lost cells (above)
            session.on_arrive(wp)
            waypoints_visited += 1
            final_ratio = ratio

    return {"total_spins": total_spins, "redundant_spins": redundant_spins,
            "waypoints_visited": waypoints_visited, "final_coverage": round(final_ratio, 4),
            "cells_lost_to_early_stop": cells_lost_to_early_stop}

## A/B run on the real + synthetic maps

`cells_lost_to_early_stop` is the key diagnostic for the hypothesis: it counts cells a heading
*would* have covered but that the arm skipped by stopping. If the `ratio` arm has a large value here
and `gain` has ~0, that is the premature-stop bug — and the `gain` arm's coverage should be ≥.

In [4]:
def _md_from_free(free, res=0.05, infl_px=1.0):
    return build_map_data((~free).astype(float), res, 0.0, 0.0, inflation_radius_m=infl_px * res)

def make_ring():
    f = np.zeros((160, 160), bool); f[20:140, 20:140] = True; f[55:105, 55:105] = False
    return _md_from_free(f)
def make_L():
    f = np.zeros((160, 160), bool); f[20:60, 20:140] = True; f[20:140, 20:60] = True
    return _md_from_free(f)
def make_multi():
    f = np.zeros((240, 240), bool)
    f[10:50, 10:50] = True; f[10:50, 190:230] = True; f[190:230, 100:140] = True
    return _md_from_free(f)
def make_open_room(n=300):
    f = np.zeros((n, n), bool); f[10:n-10, 10:n-10] = True
    return _md_from_free(f)

MAPS = {
    "lab_ghent":    load_map("lab_ghent"),
    "lab_05":       load_map("lab_05"),
    "ring":         make_ring(),
    "L":            make_L(),
    "multi-region": make_multi(),
    "open_room":    make_open_room(),
}
cfg = base_cfg()

print(f"{'map':<13}{'signal':<8}{'spins':>7}{'redund':>8}{'wps':>5}{'coverage':>10}{'lost_cells':>12}")
results = {}
for name, md in MAPS.items():
    H, W = md.pgm_array.shape
    cx, cy = W // 2, H // 2
    for sig in ("ratio", "gain"):
        m = simulate(md, cfg, sig, cx, cy)
        results[(name, sig)] = m
        print(f"{name:<13}{sig:<8}{m['total_spins']:>7}{m['redundant_spins']:>8}"
              f"{m['waypoints_visited']:>5}{m['final_coverage']*100:>9.1f}%{m['cells_lost_to_early_stop']:>12}")

print()
for name in MAPS:
    r, g = results[(name, "ratio")], results[(name, "gain")]
    d_spins = g["total_spins"] - r["total_spins"]
    d_cov = g["final_coverage"] - r["final_coverage"]
    verdict = ("gain better" if d_cov > 1e-9 and d_spins <= 0 else
               "gain better (more coverage, more spins)" if d_cov > 1e-9 else
               "tie" if abs(d_cov) < 1e-9 and d_spins == 0 else
               "gain worse" if d_cov < -1e-9 else "mixed")
    print(f"{name:<13}: Δcoverage={d_cov*100:+.2f}%  Δspins={d_spins:+d}  "
          f"ratio_lost={r['cells_lost_to_early_stop']}  gain_lost={g['cells_lost_to_early_stop']}  → {verdict}")

map          signal    spins  redund  wps  coverage  lost_cells
lab_ghent    ratio        25       0    5     18.6%           0
lab_ghent    gain         25       0    5     18.6%           0
lab_05       ratio        52       0    8     84.5%           0
lab_05       gain         52       0    8     84.5%           0
ring         ratio        22       0    5     99.9%           0
ring         gain         22       0    5     99.9%           0
L            ratio        20       0    5    100.0%           0
L            gain         20       0    5    100.0%           0
multi-region ratio         8       0    1     33.3%           0
multi-region gain          8       0    1     33.3%           0
open_room    ratio        53       1    7     97.6%           0
open_room    gain         47       0    7     97.6%        1299

lab_ghent    : Δcoverage=+0.00%  Δspins=+0  ratio_lost=0  gain_lost=0  → tie
lab_05       : Δcoverage=+0.00%  Δspins=+0  ratio_lost=0  gain_lost=0  → tie
ring         

## Outcome

_(Fill in after running.)_ Expected shape if the hypothesis holds: the `ratio` arm shows non-zero
`ratio_lost` (cells a heading would have covered but was skipped) especially on the larger maps
(open_room, lab_ghent) where the global denominator is biggest; the `gain` arm shows `gain_lost ≈ 0`
and coverage ≥ the `ratio` arm. If so, the marginal-gain stop signal is a strict improvement and
§4 should be applied via the minimal caller-side change (feed `new_cells` to `RotationState.update`).

If instead the two arms tie everywhere (global ratio never actually underflows at this map scale /
`num_rays`), then the bug is latent-only at current sizes — record that and keep §4 as a documented
known-limitation rather than changing code.